<a href="https://colab.research.google.com/github/ekaratnida/Applied-machine-learning/blob/master/Week03-MLR/Lab3-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multivariate linear regresion


In [1]:
!pip install ipython-autotime

In [2]:
%load_ext autotime

time: 351 µs (started: 2026-08-27 04:29:25 +00:00)


In [3]:
import numpy as np
np.set_printoptions(precision=2)
from sklearn.datasets import make_regression

x, y = make_regression(n_samples=100000, n_features=1000, noise=2, random_state=123)
x_b = np.c_[np.ones((x.shape[0],1)),x]


time: 16.1 s (started: 2026-08-27 04:29:25 +00:00)


In [4]:
epsilon = 0.00001
miter = 100000
eta = 0.01
# initial theta


time: 3.19 ms (started: 2026-08-27 04:29:41 +00:00)


# **Batch Gradient descent (Multiple linear regression)**


In [5]:
def cost_function(theta, x, y, N): #mse
  y_hat = x.dot(theta)
  c = (1/N)*np.sum((y_hat-y)**2)
  return c

time: 2.23 ms (started: 2026-08-27 04:29:41 +00:00)


In [8]:
import random
import math
import numpy as np # Explicitly importing numpy

def gradient_descent(alpha, x, y, ep=0.001, max_iter=10000, random_state=None):
  converged = False
  iter_count = 0
  N = x.shape[0] # number of samples

  # Set random state for reproducibility if provided
  if random_state is not None:
      rng = np.random.default_rng(random_state)
  else:
      rng = np.random.default_rng() # Use default if no seed is provided

  theta =  rng.random((x_b.shape[1],1))

  theta_history = [theta.flatten()] # Store initial theta state
  cost_history = [] # Store cost at each step

  # total error, J(theta)
  J = cost_function(theta, x, y, N)
  cost_history.append(J) # Store initial cost
  print("First J = ",J)

  # Iterate Loop
  while not converged:

    y_hat = x.dot(theta)
    diff = y_hat - y
    grad = x.T.dot(diff)

    theta = theta - alpha * (1/N) * (grad)
    theta_history.append(theta.flatten()) # Store theta at each step

    # error
    J2 = cost_function(theta, x, y, N)
    cost_history.append(J2) # Store cost at each step

    if abs(J-J2) <= ep:
        print("       Converged, iterations: ", iter_count, "/", max_iter , ", error J = ",J) # Commented out verbose printing
        converged = True

    J = J2   # update error s
    iter_count += 1  # update iter

    if iter_count == max_iter:
        print('       Max iterations exceeded!') # Commented out verbose printing
        converged = True

    if math.floor(iter_count % 100) == 0:
      print("iter = ",iter_count, ", error = ", J)

  return theta, np.array(theta_history), np.array(cost_history)

time: 4.7 ms (started: 2026-08-27 04:36:07 +00:00)


In [ ]:
from sklearn.datasets import make_regression

# Re-create a small 2D dataset for visualization, ensuring consistent variables
X_vis, y_vis = make_regression(n_samples=50, n_features=1, noise=5, random_state=42)
X_vis_b = np.c_[np.ones((X_vis.shape[0],1)), X_vis] # Add bias term
y_vis = y_vis.reshape(-1, 1)

print("X_vis_b shape:", X_vis_b.shape)
print("y_vis shape:", y_vis.shape)

# Run MBGD on the 2D simplified dataset with random_state
alpha_2d_mbgd = 0.1
ep_2d_mbgd = 1e-4 # Epsilon for 2D convergence
max_iter_2d_mbgd = 5000 # Max iterations for 2D MBGD
batch_size_2d_mbgd = 10 # Small batch size for small dataset

theta_2d_mbgd, theta_history_2d_mbgd, cost_history_2d_mbgd = mbgd(alpha_2d_mbgd, X_vis_b, y_vis, ep=ep_2d_mbgd, max_iter=max_iter_2d_mbgd, batch_size=batch_size_2d_mbgd, random_state=42)

print("MBGD 2D Theta (first 2 elements):", theta_2d_mbgd[:2].flatten())

In [ ]:
# Run BGD on the 2D simplified dataset with random_state
alpha_2d = 0.01
ep_2d = 1e-4 # Epsilon for 2D convergence
max_iter_2d = 5000 # Max iterations for 2D BGD

theta_2d, theta_history_2d, cost_history_2d = gradient_descent(alpha_2d, X_vis_b, y_vis, ep=ep_2d, max_iter=max_iter_2d, random_state=42)

print("BGD 2D Theta (first 2 elements):", theta_2d[:2].flatten())

In [ ]:
# Run SGD on the 2D simplified dataset with random_state
alpha_2d_sgd = 0.1
ep_2d_sgd = 1e-4 # Epsilon for 2D convergence
max_iter_2d_sgd = 5000 # Max iterations for 2D SGD

theta_2d_sgd, theta_history_2d_sgd, cost_history_2d_sgd = sgd(alpha_2d_sgd, X_vis_b, y_vis, ep=ep_2d_sgd, max_iter=max_iter_2d_sgd, random_state=42)

print("SGD 2D Theta (first 2 elements):", theta_2d_sgd[:2].flatten())

In [ ]:
from matplotlib import animation, rc

rc('animation', html='jshtml')

# Re-define compute_cost_2d to ensure it's available
def compute_cost_2d(X, y, theta0, theta1):
    m = len(y)
    theta_2d = np.array([[theta0], [theta1]])
    predictions = X.dot(theta_2d)
    cost = (1/(2*m)) * np.sum(np.square(predictions - y))
    return cost

# Generate a grid of theta values for the contour plot
theta0_vals = np.linspace(-10, 10, 100)
theta1_vals = np.linspace(-10, 10, 100)
J_vals = np.zeros((len(theta0_vals), len(theta1_vals)))

for i, t0 in enumerate(theta0_vals):
    for j, t1 in enumerate(theta1_vals):
        J_vals[i,j] = compute_cost_2d(X_vis_b, y_vis, t0, t1)

# Ensure J_vals is transposed for correct contour plotting
J_vals = J_vals.T

# Setup the figure and axes
fig, ax = plt.subplots(figsize=(10, 8))
ax.contour(theta0_vals, theta1_vals, J_vals, levels=np.logspace(-2, 3, 20), cmap='viridis')
ax.set_xlabel('Theta_0')
ax.set_ylabel('Theta_1')
ax.set_title('Cost Function Contour: BGD, SGD, MBGD Paths')

# Initialize the plot elements for the animation
line_bgd, = ax.plot([], [], 'o-', color='red', lw=2, markersize=4, label='BGD Path')
point_bgd, = ax.plot([], [], 'o', color='red', markersize=8)

line_sgd, = ax.plot([], [], 'o-', color='orange', lw=2, markersize=4, label='SGD Path')
point_sgd, = ax.plot([], [], 'o', color='orange', markersize=8)

line_mbgd, = ax.plot([], [], 'o-', color='purple', lw=2, markersize=4, label='MBGD Path')
point_mbgd, = ax.plot([], [], 'o', color='purple', markersize=8)

ax.legend()

# Determine the minimum length for consistent animation frames
min_frames = min(len(theta_history_2d), len(theta_history_2d_sgd), len(theta_history_2d_mbgd))

# Animation update function
def update_combined(frame):
    # BGD
    line_bgd.set_data(theta_history_2d[:frame+1, 0], theta_history_2d[:frame+1, 1])
    point_bgd.set_data([theta_history_2d[frame, 0]], [theta_history_2d[frame, 1]]) # Fixed here

    # SGD
    line_sgd.set_data(theta_history_2d_sgd[:frame+1, 0], theta_history_2d_sgd[:frame+1, 1])
    point_sgd.set_data([theta_history_2d_sgd[frame, 0]], [theta_history_2d_sgd[frame, 1]]) # Fixed here

    # MBGD
    line_mbgd.set_data(theta_history_2d_mbgd[:frame+1, 0], theta_history_2d_mbgd[:frame+1, 1])
    point_mbgd.set_data([theta_history_2d_mbgd[frame, 0]], [theta_history_2d_mbgd[frame, 1]]) # Fixed here

    return line_bgd, point_bgd, line_sgd, point_sgd, line_mbgd, point_mbgd,

# Create the animation
ani_combined = animation.FuncAnimation(fig, update_combined, frames=min_frames, interval=100, blit=True)

plt.close(fig) # Prevent the static plot from showing twice
ani_combined

In [ ]:
if __name__ == '__main__':
  print("start main")
  print(x_b.shape)
  y = y.reshape(-1,1)
  print(y.shape)
  alpha = eta
  # Updated call to gradient_descent to capture all returned values and pass random_state
  theta, theta_history_original, cost_history_original = gradient_descent(alpha, x_b, y, ep=epsilon, max_iter=miter, random_state=42)
  print ("Theta = ", theta[0:5])

start main
(100000, 1001)
(100000, 1)
First J =  30505.903681303676
iter =  100 , error =  4089.234792437116
iter =  200 , error =  574.6129761879549


# Stochastic GD
## Your turn :)

In [9]:
import random
import math
import numpy as np

def sgd(alpha, x, y, ep=0.001, max_iter=10000, random_state=None):
  converged = False
  iter_count = 0 # Renamed 'iter' to 'iter_count'
  N = x.shape[0] # number of samples

  # Set random state for reproducibility if provided
  if random_state is not None:
      rng = np.random.default_rng(random_state)
  else:
      rng = np.random.default_rng() # Use default if no seed is provided

  # initial theta
  theta =  rng.random((x.shape[1],1))

  theta_history = [theta.flatten()] # Store initial theta state
  cost_history = [] # Store cost at each step

  # total error, J(theta)
  J = cost_function(theta, x, y, N)
  print("First J = ",J)
  cost_history.append(J) # Store initial cost

  shuffle = np.random.permutation(N)
  x = x[shuffle]
  y = y[shuffle]

  rIndex = 0
  # Iterate Loop
  while not converged:

    # Select one random row index for SGD
    xr = x[rIndex].reshape(1,-1)
    y_hat = xr.dot(theta)

    diff = y_hat - y[rIndex]
    diff = diff.reshape(-1,1)
    # For SGD, gradient is typically calculated for a single sample, not averaged over N
    grad = xr.T.dot(diff)

    # Update theta. Note: Some SGD implementations might not divide by N here.
    # We'll keep it consistent with the original batch GD scaling for this example.
    theta = theta - alpha * (grad)
    theta_history.append(theta.flatten()) # Store theta at each step

    alpha = alpha/(1+(alpha*iter_count)) # Use iter_count for alpha decay

    # error
    J2 = cost_function(theta, x, y, N)
    #print("J2 = ",J2)
    cost_history.append(J2) # Store cost at each step

    if abs(J-J2) <= ep:
        converged = True
        print("       Converged, iterations: ", iter_count, "/", max_iter, ", error J = ",J)

    J = J2   # update error s
    iter_count += 1  # update iter

    if iter_count == max_iter:
      converged = True
      print('       Max iterations exceeded!')

    if math.floor(iter_count % 100) == 0:
      print("iter = ",iter_count, ", error = ", J)

    rIndex += 1
    if rIndex >= N:
      rIndex = 0
      shuffle = np.random.permutation(N)
      x = x[shuffle]
      y = y[shuffle]

  return theta, np.array(theta_history), np.array(cost_history)

time: 3.91 ms (started: 2026-08-27 04:36:44 +00:00)


In [ ]:
if __name__ == '__main__':
  print("start main")
  print(x_b.shape)
  y = y.reshape(-1,1)
  print(y.shape)
  alpha = eta
  # Updated call to sgd to capture all returned values and pass random_state
  theta, theta_history_original_sgd, cost_history_original_sgd = sgd(alpha, x_b, y, ep=epsilon, max_iter=miter, random_state=42)
  print ("Theta = ", theta[0:5])

# Mini-batch GD (b = 1000)
## Your turn :)

In [10]:
import random
import math
import numpy as np

def mbgd(alpha, x, y, ep=0.001, max_iter=10000, batch_size=1000, random_state=None):
  converged = False
  iter_count = 0
  N = x.shape[0] # total number of samples

  # Set random state for reproducibility if provided
  if random_state is not None:
      rng = np.random.default_rng(random_state)
  else:
      rng = np.random.default_rng() # Use default if no seed is provided

  # initial theta
  theta = rng.random((x.shape[1],1))

  theta_history = [theta.flatten()] # Store initial theta state
  cost_history = [] # Store cost at each step

  # total error, J(theta) - calculated on the full dataset for monitoring
  J = cost_function(theta, x, y, N)
  print("First J = ",J)
  cost_history.append(J) # Store initial cost

  # Main loop
  while not converged:

    # Shuffle data at the start of each epoch
    if iter_count % (N // batch_size) == 0: # Check if a new epoch starts
        shuffle_indices = np.random.permutation(N)
        x_shuffled = x[shuffle_indices]
        y_shuffled = y[shuffle_indices]

    # Get current mini-batch
    start_idx = (iter_count * batch_size) % N
    end_idx = min(start_idx + batch_size, N)

    # Handle the case where the last batch might be smaller
    x_batch = x_shuffled[start_idx:end_idx]
    y_batch = y_shuffled[start_idx:end_idx]
    N_batch = x_batch.shape[0]

    y_hat_batch = x_batch.dot(theta)
    diff_batch = y_hat_batch - y_batch
    grad = x_batch.T.dot(diff_batch)

    # Update theta. Divide by batch_size for proper averaging of gradients.
    theta = theta - alpha * (1/N_batch) * (grad)
    theta_history.append(theta.flatten()) # Store theta at each step

    # Learning rate decay (can be adjusted for MBGD)
    alpha = alpha/(1+(alpha*0.01*iter_count)) # Slightly different decay factor

    # Error (cost) - calculated on the full dataset for consistent monitoring
    J2 = cost_function(theta, x, y, N)
    cost_history.append(J2) # Store cost at each step

    if abs(J-J2) <= ep:
        converged = True
        print("       Converged, iterations: ", iter_count, "/", max_iter, ", error J = ",J)

    J = J2   # update error s
    iter_count += 1  # update iter

    if iter_count == max_iter:
      converged = True
      print('       Max iterations exceeded!')

    # Optional: Print progress every X iterations (e.g., every 100 mini-batch updates)
    if math.floor(iter_count % 100) == 0:
      print("iter = ",iter_count, ", error = ", J)

  return theta, np.array(theta_history), np.array(cost_history)

time: 4.06 ms (started: 2026-08-27 04:36:45 +00:00)


In [ ]:
if __name__ == '__main__':
  print("start main for Mini-batch GD")
  print(x_b.shape)
  y_reshaped = y.reshape(-1,1) # Ensure y is reshaped if not already
  print(y_reshaped.shape)

  alpha = eta # Learning rate for MBGD
  batch_size = 1000 # As suggested by the text cell

  # Run Mini-batch Gradient Descent and pass random_state
  theta_mbgd, theta_history_original_mbgd, cost_history_original_mbgd = mbgd(alpha, x_b, y_reshaped, ep=epsilon, max_iter=miter, batch_size=batch_size, random_state=42)

  print ("Theta (first 5 elements) = ", theta_mbgd[0:5])

### Mini-batch Gradient Descent Cost History

### Comparison of Cost Histories: BGD, SGD, and MBGD

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 7))

# Plot BGD Cost History
plt.plot(range(len(cost_history_original)), cost_history_original, color='blue', label='Batch Gradient Descent')

# Plot SGD Cost History
plt.plot(range(len(cost_history_original_sgd)), cost_history_original_sgd, color='orange', label='Stochastic Gradient Descent')

# Plot MBGD Cost History
plt.plot(range(len(cost_history_original_mbgd)), cost_history_original_mbgd, color='purple', label=f'Mini-batch Gradient Descent (batch_size={batch_size})')

plt.xlabel('Iterations')
plt.ylabel('Cost (J)')
plt.title('Cost History Comparison: BGD, SGD, and MBGD')
plt.legend()
plt.grid(True)
plt.ylim(bottom=0, top=cost_history_original_sgd.max()*1.1) # Adjust y-limit to better visualize differences
plt.show()

In [ ]:
print("Batch Gradient Descent (BGD) first 5 theta:")
print(theta_history_original[-1, :5])

print("\nStochastic Gradient Descent (SGD) first 5 theta:")
print(theta_history_original_sgd[-1, :5])

print("\nMini-batch Gradient Descent (MBGD) first 5 theta:")
print(theta_mbgd[:5])

# Polynomial regression
Reference: https://towardsdatascience.com/machine-learning-polynomial-regression-with-python-5328e4e8a386

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Importing the dataset
df = pd.read_csv('https://raw.githubusercontent.com/ekaratnida/Applied-machine-learning/master/Week03-MLR/Position_Salaries.csv')
X = df.iloc[:, 1:2].values
y = df.iloc[:, 2].values
print(df.head())

In [ ]:
# Fitting Linear Regression to the dataset
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(X, y)
print(y)
# Visualizing the Linear Regression results

plt.scatter(X, y, color='red')
plt.plot(X, lin_reg.predict(X), color='blue')
plt.title('Truth or Bluff (Linear Regression)')
plt.xlabel('Position level')
plt.ylabel('Salary')
plt.show()


In [ ]:
# Fitting Polynomial Regression to the dataset
from sklearn.preprocessing import PolynomialFeatures
poly_reg = PolynomialFeatures(degree=3)
X_poly = poly_reg.fit_transform(X)
print(X_poly)
pol_reg = LinearRegression()
pol_reg.fit(X_poly, y)

# Visualizing the Polymonial Regression results
plt.figure(figsize=(6,4))
plt.scatter(X, y, color='red')
plt.plot(X, lin_reg.predict(X), color='green')
plt.plot(X, pol_reg.predict(X_poly), color='blue')
plt.title('Truth or Bluff (Linear Regression)')
plt.xlabel('Position level')
plt.ylabel('Salary')
plt.show()


In [ ]:
# Predicting a new result with Linear Regression
print(lin_reg.predict([[5.5]]))

# Predicting a new result with Polymonial Regression
print(pol_reg.predict(poly_reg.fit_transform([[5.5]])))